# Figure 3h,i — Axon meshes colour-coded by synapse density

The neuron's surface mesh is coloured by local synapse density, showing that synapses
concentrate on axon terminals rather than spreading along the arbor.

| Panel | Synapses counted | Query direction |
| --- | --- | --- |
| **h** | axodendritic | sensory neuron **presynaptic** (`pre`) |
| **i** | axoaxonic | sensory neuron **postsynaptic** (`post`) |

**Method** (Methods, *Synapse-density coloring of neuron meshes*). The mesh is turned into a
vertex-adjacency graph with edge weights equal to Euclidean distance between neighbouring
vertices. Each synapse is snapped to its nearest vertex, and a **geodesic** neighbourhood
of radius **R = 4 µm** is found by Dijkstra search over that graph — distance along the
membrane, not straight-line — with every vertex inside the cutoff incremented by one.
Summing over all synapses gives a per-vertex density, computed separately for the two
directions.

Both panels share one `viridis` scale capped at **15 synapses**, so the axodendritic and
axoaxonic maps are directly comparable; a matching colourbar is exported alongside.

**Output.** Colour-baked `.ply` meshes (rendered externally) plus the colourbar SVG.
The published panels use the Aδ-LTMR `720575940893392149`; the other seven subtypes are
listed commented-out in the config and can be enabled.

**Requires CAVE** — the mesh is fetched through CloudVolume and synapses queried live.
Meshes are far too large to ship, so there is no offline path for this panel.


In [ ]:
# ===============================================
# Synapse-density coloring using shared VIRIDIS scale
# - PRE and POST use the same heatmap
# - Shared cap is user-controlled
# - Exports colored PLY meshes
# - Optionally saves capped SVG heatmap/colorbar
# ===============================================

import os
from typing import Optional, Dict, Tuple

import numpy as np
from caveclient import CAVEclient
import cloudvolume as cv
import trimesh
import networkx as nx

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize


# -----------------------------
# Configuration
# -----------------------------
root_ids = [
 #  720575940922958792, # cltmr
#   720575940920833705, # mrgd
   720575940893392149, # ad-ltmr
 #  720575940881926778, # ab-ltmr
#   720575940882155492, # sst
#   720575940847552365, # ad-htmr
#   720575940881918801, # chtmr-pep
#   720575940927175960, # cold
]

DATASTACK = "wclee_mouse_spinalcord_cltmr"
SEG_URL = "graphene://https://cave.fanc-fly.com/segmentation/table/wclee_mouse_spinalcord_cltmr"

CV_SECRETS: Optional[str] = None

SYN_TABLE = "sensory_neuron_synapses"
RES_NM_PER_VOX = np.array([16.0, 16.0, 45.0], dtype=float)

R = 4000.0

OUTPUT_DIR = "figure3hi_mesh_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# -----------------------------
# Shared heatmap controls
# -----------------------------
COLORMAP_NAME = "viridis"

# Main control:
# Any density >= DENSITY_CAP gets the brightest viridis color.
DENSITY_CAP = 15.0

# Save one shared capped heatmap/colorbar?
SAVE_COLORBAR = True


# -----------------------------
# Helpers
# -----------------------------
def get_scaled_synapses(client: CAVEclient,
                        seg_id: int,
                        query_type: str,
                        syn_table: str,
                        mat_version: int,
                        res_nm_per_vox: np.ndarray) -> np.ndarray:
    """
    query_type: 'pre' or 'post'
    Returns Nx3 synapse centroid positions in nm.
    """
    df = client.materialize.synapse_query(
        **{f"{query_type}_ids": [seg_id]},
        synapse_table=syn_table,
        materialization_version=mat_version,
        split_positions=True
    )

    if df.empty:
        return np.empty((0, 3), dtype=float)

    pts = np.stack([
        np.hstack(df["ctr_pt_position_x"].values),
        np.hstack(df["ctr_pt_position_y"].values),
        np.hstack(df["ctr_pt_position_z"].values)
    ], axis=1).astype(float)

    return pts * res_nm_per_vox


def build_vertex_graph_with_weights(t_mesh: trimesh.Trimesh) -> nx.Graph:
    """
    NetworkX graph with edge weights equal to Euclidean edge length.
    """
    G = t_mesh.vertex_adjacency_graph
    verts = t_mesh.vertices

    for u, v in G.edges:
        G[u][v]["weight"] = float(np.linalg.norm(verts[u] - verts[v]))

    return G


def density_from_synapses_on_surface(t_mesh: trimesh.Trimesh,
                                     syn_pts_nm: np.ndarray,
                                     G: nx.Graph,
                                     radius_nm: float) -> np.ndarray:
    """
    For each synapse:
      1. Snap to nearest mesh vertex.
      2. Run geodesic Dijkstra search within radius_nm.
      3. Increment all reached vertices.
    """
    n_verts = t_mesh.vertices.shape[0]

    if syn_pts_nm.shape[0] == 0:
        return np.zeros(n_verts, dtype=float)

    prox = trimesh.proximity.ProximityQuery(t_mesh)
    _, closest_vertices = prox.vertex(syn_pts_nm)

    density = np.zeros(n_verts, dtype=float)

    unique_idx, counts = np.unique(closest_vertices, return_counts=True)

    for syn_vtx, mult in zip(unique_idx, counts):
        g_dists = nx.single_source_dijkstra_path_length(
            G,
            int(syn_vtx),
            cutoff=radius_nm,
            weight="weight"
        )

        for v in g_dists.keys():
            density[v] += float(mult)

    return density


def _safe_concat(dct: Dict[int, np.ndarray]) -> np.ndarray:
    arrs = []

    for a in dct.values():
        a = np.asarray(a, dtype=float)
        a = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
        arrs.append(a)

    if len(arrs) == 0:
        return np.array([], dtype=float)

    return np.concatenate(arrs)


def save_colorbar_svg(cmap,
                      norm: Normalize,
                      title: str,
                      svg_path: str) -> None:
    """
    Saves a horizontal SVG colorbar for the shared capped heatmap.
    """
    fig, ax = plt.subplots(figsize=(8, 1.2))

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=ax, orientation="horizontal")

    vmin, vmax = float(norm.vmin), float(norm.vmax)
    ticks = np.linspace(vmin, vmax, 6)
    tick_labels = [f"{t:.0f}" for t in ticks]

    cbar.set_ticks(ticks)
    cbar.set_ticklabels(tick_labels)
    cbar.set_label(title, fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    plt.savefig(svg_path, format="svg", dpi=300, bbox_inches="tight")
    plt.close(fig)


def export_colored_mesh(mesh: trimesh.Trimesh,
                        density: np.ndarray,
                        cmap,
                        norm: Normalize,
                        density_cap: float,
                        out_path: str) -> None:
    """
    Clips density, maps to RGB with cmap/norm, and exports PLY.
    """
    density = np.asarray(density, dtype=float)
    density = np.nan_to_num(density, nan=0.0, posinf=0.0, neginf=0.0)

    density_clipped = np.clip(density, 0.0, density_cap)

    rgba = cmap(norm(density_clipped))
    vertex_colors = (rgba[:, :3] * 255).astype(np.uint8)

    t_mesh = trimesh.Trimesh(
        vertices=mesh.vertices,
        faces=mesh.faces,
        vertex_colors=vertex_colors,
        process=False
    )

    t_mesh.visual.vertex_colors = vertex_colors

    with open(out_path, "wb") as f:
        f.write(trimesh.exchange.ply.export_ply(t_mesh))


# -----------------------------
# Main build
# -----------------------------
client = CAVEclient(DATASTACK)
# Pin the materialization so the panel is reproducible.
MATERIALIZATION = 669
mat_version = (MATERIALIZATION if MATERIALIZATION is not None
               else client.materialize.most_recent_version())

pre_density_all: Dict[int, np.ndarray] = {}
post_density_all: Dict[int, np.ndarray] = {}
mesh_all: Dict[int, trimesh.Trimesh] = {}

for root_id in root_ids:
    print(f"Processing root_id: {root_id}")

    seg_id = client.chunkedgraph.get_latest_roots(root_id)[0]

    vol = cv.CloudVolume(
        SEG_URL,
        secrets=CV_SECRETS,
        use_https=True,
        progress=False
    )

    cv_mesh = vol.mesh.get(seg_id)

    if seg_id not in cv_mesh:
        print(f"WARNING: seg_id {seg_id} mesh not found; skipping")
        continue

    mesh_obj = cv_mesh[seg_id]

    base_mesh = trimesh.Trimesh(
        vertices=mesh_obj.vertices,
        faces=mesh_obj.faces,
        process=False
    )

    G = build_vertex_graph_with_weights(base_mesh)

    pre_pts_nm = get_scaled_synapses(
        client,
        seg_id,
        "pre",
        SYN_TABLE,
        mat_version,
        RES_NM_PER_VOX
    )

    post_pts_nm = get_scaled_synapses(
        client,
        seg_id,
        "post",
        SYN_TABLE,
        mat_version,
        RES_NM_PER_VOX
    )

    pre_density = density_from_synapses_on_surface(
        base_mesh,
        pre_pts_nm,
        G,
        R
    )

    post_density = density_from_synapses_on_surface(
        base_mesh,
        post_pts_nm,
        G,
        R
    )

    pre_density_all[root_id] = pre_density
    post_density_all[root_id] = post_density
    mesh_all[root_id] = base_mesh


# -----------------------------
# Shared normalization
# -----------------------------
all_pre = _safe_concat(pre_density_all)
all_post = _safe_concat(post_density_all)

max_pre_density = float(all_pre.max()) if all_pre.size else 0.0
max_post_density = float(all_post.max()) if all_post.size else 0.0
max_shared_density = max(max_pre_density, max_post_density)

cmap_shared = cm.get_cmap(COLORMAP_NAME)
norm_shared = Normalize(vmin=0.0, vmax=DENSITY_CAP)

print("Observed max_pre_density:", max_pre_density)
print("Observed max_post_density:", max_post_density)
print("Observed max_shared_density:", max_shared_density)
print("DENSITY_CAP used:", DENSITY_CAP)


# -----------------------------
# Export colored PRE/POST meshes
# -----------------------------
cap_label = str(DENSITY_CAP).replace(".", "p")

for root_id in root_ids:
    if root_id not in mesh_all:
        continue

    mesh = mesh_all[root_id]

    if root_id in pre_density_all:
        out_path = os.path.join(
            OUTPUT_DIR,
            f"{root_id}_pre_{COLORMAP_NAME}_sharedCap{cap_label}.ply"
        )

        export_colored_mesh(
            mesh=mesh,
            density=pre_density_all[root_id],
            cmap=cmap_shared,
            norm=norm_shared,
            density_cap=DENSITY_CAP,
            out_path=out_path
        )

    if root_id in post_density_all:
        out_path = os.path.join(
            OUTPUT_DIR,
            f"{root_id}_post_{COLORMAP_NAME}_sharedCap{cap_label}.ply"
        )

        export_colored_mesh(
            mesh=mesh,
            density=post_density_all[root_id],
            cmap=cmap_shared,
            norm=norm_shared,
            density_cap=DENSITY_CAP,
            out_path=out_path
        )


# -----------------------------
# Optionally save capped shared heatmap/colorbar
# -----------------------------
if SAVE_COLORBAR:
    colorbar_path = os.path.join(
        OUTPUT_DIR,
        f"colorbar_PRE_POST_{COLORMAP_NAME}_sharedCap{cap_label}.svg"
    )

    save_colorbar_svg(
        cmap_shared,
        norm_shared,
        title=f"Synapse density, PRE/POST shared scale clipped at {DENSITY_CAP:g}",
        svg_path=colorbar_path
    )


print("Done.")
print("Output dir:", OUTPUT_DIR)